# Exercise C — Filter by Section Before Retrieving
Tag each chunk with the PDF section it came from, then restrict retrieval to one section. This kills cross-section noise — the Chroma Studio filtering idea, applied to RAG.

In [ ]:
# Offline mock so this scaffold runs with NO API key / NO network.
# For real practice, replace `embed()` with your real embedder (InHouseEmbeddings,
# SentenceTransformer, etc.) and `llm()` with a real model call.
import numpy as np, re
_STOP=set("the a an to of and or is are be for in on at by with from as that this it its".split())
def _tok(t): return [w for w in re.findall(r"[a-z0-9]+",t.lower()) if w not in _STOP and len(w)>2]
def embed(texts):
    if isinstance(texts,str): texts=[texts]
    out=[]
    for t in texts:
        v=np.zeros(256)
        for w in _tok(t): v[abs(hash(w))%256]+=1
        n=np.linalg.norm(v); out.append(v/n if n else v)
    return np.array(out)
def cos(a,b): return float(a@b)

# A small corpus standing in for chunks of ERP-2008-chapter4.pdf (health-care economics).
CORPUS = [
 ("Demand for health care is derived from the value of improved health, not the procedures themselves.","demand"),
 ("Health can be defined by longevity (length of life) and quality of life.","demand"),
 ("National health spending reached over 7000 dollars per capita and about 16 percent of GDP.","spending"),
 ("Medical technology accounts for about half of long-term health spending growth.","spending"),
 ("Medicare, enacted in 1965, covers people aged 65 and older; Part D is the drug benefit.","medicare"),
 ("Medicaid, established in 1965, is a program for low-income individuals, administered by states.","medicaid"),
 ("Moral hazard is the tendency to overuse care when insurance covers most of the cost.","moral_hazard"),
 ("Adverse selection is when insurance is most attractive to those most likely to need it.","insurance"),
 ("Health Savings Accounts use pre-tax dollars with high-deductible plans to reduce routine-care reliance.","hsa"),
 ("The proposed standard deduction for health insurance would be a flat 15000 dollars per family.","tax"),
]
texts=[c[0] for c in CORPUS]; sections=[c[1] for c in CORPUS]
print("Mock corpus ready:", len(texts), "chunks.")

In [ ]:
# Our mock corpus already carries a `sections` label per chunk.
qv = embed("who does this program cover?")[0]

def retrieve(query_vec, where_section=None, k=3):
    idxs = range(len(texts))
    if where_section:
        idxs = [i for i in idxs if sections[i]==where_section]
    scored = sorted(idxs, key=lambda i: -cos(query_vec, embed(texts[i])[0]))[:k]
    return [(texts[i], sections[i]) for i in scored]

print("UNFILTERED — 'who does this program cover?'")
for t,s in retrieve(qv): print(f"  [{s}] {t[:70]}")

print("\nFILTERED to section='medicare':")
for t,s in retrieve(qv, where_section="medicare"): print(f"  [{s}] {t[:70]}")

### Observe & decide
- Unfiltered, 'who does this cover?' might pull Medicare AND Medicaid AND insurance. Filtering to `medicare` scopes it precisely.
**Your turn:** in the real notebook, add a `section` metadata field at ingest and use Chroma's `where={'section': ...}` before semantic search. Then open the collection in Chroma Studio and filter the same way visually.